# Scraping Tramite com Selenium

Fluxo: abrir navegador, logar, extrair campos gerais do processo e salvar em Excel.


In [ ]:
# INSTALAÇÃO AUTOMÁTICA (execute esta célula uma única vez e a exclua do notebook)
import sys
import subprocess

def run(cmd):
    print('>>', ' '.join(cmd))
    subprocess.check_call(cmd)

# Atualiza ferramentas basicas de empacotamento
run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])

# Instala bibliotecas usadas no notebook
run([sys.executable, '-m', 'pip', 'install', 'selenium', 'pandas', 'openpyxl', 'ipykernel'])

print('Instalacao concluida. Reinicie o kernel do notebook e execute as celulas em seguida.')
print('Obs.: o Google Chrome deve estar instalado no computador para o Selenium abrir o navegador.')


>> C:\Users\tc832880\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip setuptools wheel
>> C:\Users\tc832880\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install selenium pandas openpyxl ipykernel
Instalacao concluida. Reinicie o kernel do notebook e execute as celulas em seguida.
Obs.: o Google Chrome deve estar instalado no computador para o Selenium abrir o navegador.


In [2]:
import re
import time
import subprocess
import unicodedata
from dataclasses import dataclass, asdict
from typing import Optional

import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [ ]:
# ========= CONFIGURAÇÃO DE LINKS DOS PROCESSOS E CREDENCIAIS DE USUÁRIO =========


LINKS_PROCESSO = [
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=202696070",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026122855",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=202696746",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026107295",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026113350",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026116065",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026116073",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026118963",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026122855",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026122693",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026123886",
    "https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=202696070",
]

USUARIO = "tc000000"
SENHA = "senha123TCE@#"

PASTA_EXCEL_SINCRONIZADA = "C:\\Users\\tc832880\\Tribunal de Contas do Estado do Paran\u00e1\\CGF 2025 26 - TramiteScraping"
ARQUIVO_EXCEL = f"{PASTA_EXCEL_SINCRONIZADA}\\processos_tramite_cacs.xlsx"

ABA_EXCEL = "dados"
TEMPO_ESPERA = 30

FECHAR_NAVEGADOR_AUTO = True
MAX_TENTATIVAS_CARREGAMENTO = 8  # recarrega link em caso de erro 500 ou campos vazios


In [ ]:
# --- FUNÇÕES UTILITÁRIAS ---


@dataclass
class ResultadoExtracao:
    link_processo: str
    processo: Optional[str] = None
    assunto_subassunto: Optional[str] = None
    entidade: Optional[str] = None
    autuacao: Optional[str] = None


def norm_texto(texto: str) -> str:
    txt = "".join(ch for ch in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", txt).strip().lower()


def extrair_primeira_data(texto: str) -> Optional[str]:
    m = re.search(r"\b\d{2}/\d{2}/\d{4}\b", texto or '')
    return m.group(0) if m else None


def obter_texto_bloco_informacoes(driver) -> str:
    xpaths = [
        "//*[contains(., 'Assunto / Subassunto') and contains(., 'Autua??o')]",
        "//*[contains(., 'Assunto / Subassunto') and contains(., 'Autuacao')]",
    ]
    melhor = ''
    for xp in xpaths:
        for e in driver.find_elements(By.XPATH, xp):
            try:
                if not e.is_displayed():
                    continue
                t = (e.text or '').strip()
                if len(t) > len(melhor):
                    melhor = t
            except Exception:
                pass
    return melhor if melhor else driver.find_element(By.TAG_NAME, 'body').text


def extrair_valor_box_label(driver, label: str) -> Optional[str]:
    candidatos = [
        f"//*[normalize-space()='{label}']",
        f"//label[normalize-space()='{label}']",
        f"//*[contains(normalize-space(),'{label}')]",
    ]
    for xp in candidatos:
        elems = driver.find_elements(By.XPATH, xp)
        for el in elems[:20]:
            try:
                if not el.is_displayed():
                    continue
                prox_inputs = el.find_elements(By.XPATH, "./following::*[self::input or self::textarea or self::select][1]")
                if prox_inputs:
                    v = (prox_inputs[0].get_attribute('value') or '').strip()
                    if v:
                        return v
                container = None
                try:
                    container = el.find_element(By.XPATH, "ancestor::fieldset[1]")
                except Exception:
                    try:
                        container = el.find_element(By.XPATH, "ancestor::div[1]")
                    except Exception:
                        container = None
                if container is not None:
                    lines = [ln.strip() for ln in (container.text or '').splitlines() if ln.strip()]
                    lines = [ln for ln in lines if norm_texto(ln) != norm_texto(label)]
                    for ln in lines:
                        if ln and norm_texto(ln) not in ['processo','assunto / subassunto','entidade','autuacao','autua??o']:
                            return ln
            except Exception:
                pass
    return None


def valor_por_label_texto(texto: str, label_regex: str) -> Optional[str]:
    p1 = rf"{label_regex}\s*\n\s*(.+)"
    m1 = re.search(p1, texto, flags=re.IGNORECASE)
    if m1:
        return m1.group(1).strip()
    p2 = rf"{label_regex}\s*[:\-]\s*(.+)"
    m2 = re.search(p2, texto, flags=re.IGNORECASE)
    return m2.group(1).strip() if m2 else None


def extrair_processo_robusto(texto: str) -> Optional[str]:
    for p in [r"Processo\s*\n\s*([^\n]+)", r"Processo\s*[:\-]\s*([^\n]+)"]:
        m = re.search(p, texto, flags=re.IGNORECASE)
        if m:
            cand = m.group(1).strip()
            if re.search(r"\d", cand) and '/' in cand:
                return cand
    m2 = re.search(r"\b\d{3,7}-\d+/\d{2}\b", texto)
    if m2:
        return m2.group(0)
    return None


def extrair_autuacao_robusta(texto: str) -> Optional[str]:
    m = re.search(r"Autua[c?][a?]o\s*\n\s*(\d{2}/\d{2}/\d{4})", texto, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    return extrair_primeira_data(texto)



def pagina_com_erro_500(texto_pagina: str) -> bool:
    t = norm_texto(texto_pagina or '')
    return (
        '500 ops' in t
        or 'ocorreu um erro' in t
        or 'estamos trabalhando para resolver este problema' in t
    )


def campos_minimos_ok(resultado) -> bool:
    # Considera carregado quando os campos principais estiverem preenchidos
    return bool(resultado.processo and resultado.assunto_subassunto and resultado.entidade)


def encerrar_driver(driver) -> None:
    pid = None
    try:
        pid = getattr(getattr(driver, 'service', None), 'process', None)
        pid = getattr(pid, 'pid', None)
    except Exception:
        pid = None

    try:
        driver.quit()
        time.sleep(1)
        print('Driver finalizado via quit().')
    except Exception as e:
        print('Falha no quit():', e)

    # fallback Windows: encerra arvore do chromedriver
    if pid:
        try:
            subprocess.run(['taskkill', '/PID', str(pid), '/T', '/F'], check=False, capture_output=True, text=True)
            print(f'Fallback taskkill executado para PID {pid}.')
        except Exception as e:
            print('Falha no fallback taskkill:', e)


In [ ]:
# --- Etapa 1: iniciar navegador e abrir primeiro link ---

links_validos = [l.strip() for l in LINKS_PROCESSO if l and 'COLE_AQUI' not in l]
if not links_validos:
    raise RuntimeError('Preencha ao menos 1 link em LINKS_PROCESSO.')

service = Service()
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, TEMPO_ESPERA)

try:
    driver.maximize_window()
    driver.set_window_rect(x=0, y=0, width=1920, height=1080)
except Exception:
    pass

print('Abrindo primeiro link do processo...')
driver.get(links_validos[0])
wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
time.sleep(2)
print('Tela inicial carregada.')


Abrindo primeiro link do processo...
Tela inicial carregada.


In [ ]:
# --- Etapa 2: login ---

def preencher_primeiro(seletores, valor):
    for by, sel in seletores:
        elems = driver.find_elements(by, sel)
        if elems:
            elems[0].clear()
            elems[0].send_keys(valor)
            return True
    return False


def clicar_primeiro(seletores):
    for by, sel in seletores:
        elems = driver.find_elements(by, sel)
        if elems:
            driver.execute_script('arguments[0].click();', elems[0])
            return True
    return False


def fazer_login() -> None:
    sel_user = [
        (By.CSS_SELECTOR, "input[placeholder*='Matr?cula']"),
        (By.CSS_SELECTOR, "input[placeholder*='Matr?cula']"),
        (By.CSS_SELECTOR, "input[placeholder*='CPF']"),
        (By.CSS_SELECTOR, "input[name='username']"),
        (By.CSS_SELECTOR, "input[name='usuario']"),
        (By.CSS_SELECTOR, "input[type='text']"),
    ]

    sel_senha = [
        (By.CSS_SELECTOR, "input[placeholder*='Senha']"),
        (By.CSS_SELECTOR, "input[name='password']"),
        (By.CSS_SELECTOR, "input[name='senha']"),
        (By.CSS_SELECTOR, "input[type='password']"),
    ]

    sel_botao = [
        (By.XPATH, "//button[contains(., 'Entrar')]"),
        (By.CSS_SELECTOR, "button[type='submit']"),
        (By.CSS_SELECTOR, "input[type='submit']"),
    ]

    ok_user = preencher_primeiro(sel_user, USUARIO)
    ok_senha = preencher_primeiro(sel_senha, SENHA)
    ok_btn = clicar_primeiro(sel_botao)

    print('Login - usuario:', ok_user, '| senha:', ok_senha, '| botao:', ok_btn)
    if not (ok_user and ok_senha and ok_btn):
        raise RuntimeError('Nao foi possivel preencher/clicar no login. Ajuste os seletores.')

    time.sleep(8)
    print('Login executado. URL atual:', driver.current_url)


# faz login uma vez no primeiro acesso
fazer_login()


Login - usuario: True | senha: True | botao: True
Login executado. URL atual: https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=202696070


In [ ]:
# --- Etapa 3: extrair campos para todos os links (com recarga em erro 500) ---

resultados = []

for i, link in enumerate(links_validos, start=1):
    print(f'\n[{i}/{len(links_validos)}] Processando: {link}')

    sucesso_link = False

    for tentativa in range(1, MAX_TENTATIVAS_CARREGAMENTO + 1):
        print(f'  Tentativa {tentativa}/{MAX_TENTATIVAS_CARREGAMENTO}...')

        driver.get(link)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        time.sleep(2)

        # se voltou para login, autentica e recarrega link
        if driver.find_elements(By.CSS_SELECTOR, "input[type='password']"):
            print('  Tela de login detectada novamente. Reautenticando...')
            fazer_login()
            driver.get(link)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
            time.sleep(2)

        texto_pagina = driver.find_element(By.TAG_NAME, 'body').text
        if pagina_com_erro_500(texto_pagina):
            print('  Erro 500 detectado. Recarregando pagina...')
            driver.refresh()
            time.sleep(3)
            continue

        texto_info = obter_texto_bloco_informacoes(driver)

        resultado = ResultadoExtracao(link_processo=link)

        proc_box = extrair_valor_box_label(driver, 'Processo')
        assunto_box = extrair_valor_box_label(driver, 'Assunto / Subassunto')
        entidade_box = extrair_valor_box_label(driver, 'Entidade')
        autuacao_box = extrair_valor_box_label(driver, 'Autua??o') or extrair_valor_box_label(driver, 'Autuacao')

        resultado.processo = proc_box if proc_box and re.search(r'\d', proc_box) else extrair_processo_robusto(texto_info)
        resultado.assunto_subassunto = assunto_box or valor_por_label_texto(texto_info, r'Assunto\s*/\s*Subassunto')
        resultado.entidade = entidade_box or valor_por_label_texto(texto_info, r'Entidade')
        resultado.autuacao = extrair_primeira_data(autuacao_box or '') or extrair_autuacao_robusta(texto_info)

        if not campos_minimos_ok(resultado):
            print('  Campos principais ainda vazios. Recarregando pagina...')
            driver.refresh()
            time.sleep(3)
            continue

        resultados.append(asdict(resultado))
        sucesso_link = True

        print('  Processo:', resultado.processo)
        print('  Assunto/Subassunto:', resultado.assunto_subassunto)
        print('  Entidade:', resultado.entidade)
        print('  Autuacao:', resultado.autuacao)
        break

    if not sucesso_link:
        print('  Nao foi possivel carregar dados desse link apos varias tentativas. Pulando para o proximo.')

if not resultados:
    raise RuntimeError('Nenhum resultado foi extraido.')



[1/12] Processando: https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=202696070
  Tentativa 1/8...
  Processo: 9607-0/26
  Assunto/Subassunto: REQUERIMENTO EXTERNO
  Entidade: MUNICÍPIO DE CORONEL VIVIDA
  Autuacao: 13/02/2026

[2/12] Processando: https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026122855
  Tentativa 1/8...
  Processo: 12285-5/26
  Assunto/Subassunto: REQUERIMENTO EXTERNO
  Entidade: MUNICÍPIO DE IBEMA
  Autuacao: 26/02/2026

[3/12] Processando: https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=202696746
  Tentativa 1/8...
  Processo: 9674-6/26
  Assunto/Subassunto: REQUERIMENTO EXTERNO
  Entidade: MUNICÍPIO DE VERA CRUZ DO OESTE
  Autuacao: 13/02/2026

[4/12] Processando: https://servicos.tce.pr.gov.br/TCEPR/Tribunal/TramiteWeb/Protocolo?nrProtocolo=2026107295
  Tentativa 1/8...
  Processo: 10729-5/26
  Assunto/Subassunto: REQUERIMENTO EXTERNO
  Entidade: MUNICÍPIO DE ARAPONGAS
 

In [12]:
# --- Etapa 4 removida (conclusão CACS) ---

print('Campo de conclusão CACS será preenchido manualmente no Excel.')


Campo de conclusão CACS será preenchido manualmente no Excel.


In [ ]:
# --- Etapa 5: salvar no Excel sem duplicar link_processo (dedupe robusto) ---

from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

def normalizar_link(url: str) -> str:
    if url is None:
        return ''
    u = str(url).strip()
    if not u or u.lower() in {'nan', 'none'}:
        return ''

    try:
        p = urlsplit(u)
        scheme = (p.scheme or 'https').lower()
        netloc = p.netloc.lower()
        path = (p.path or '').rstrip('/')
        q = parse_qsl(p.query, keep_blank_values=True)
        query = urlencode(sorted(q)) if q else ''
        return urlunsplit((scheme, netloc, path, query, ''))
    except Exception:
        return u.lower().rstrip('/')


novos = pd.DataFrame(resultados)
if novos.empty:
    raise RuntimeError('Nenhum resultado para salvar.')

if 'link_processo' not in novos.columns:
    raise RuntimeError('Coluna link_processo não encontrada nos novos dados.')

# garante coluna manual
if 'conclusao_cacs' not in novos.columns:
    novos['conclusao_cacs'] = ''

novos['link_processo'] = novos['link_processo'].astype(str).str.strip()
novos['link_norm'] = novos['link_processo'].apply(normalizar_link)
novos = novos[novos['link_norm'] != ''].copy()
novos = novos.drop_duplicates(subset=['link_norm'], keep='first').copy()

try:
    existente = pd.read_excel(ARQUIVO_EXCEL, sheet_name=ABA_EXCEL)
except Exception:
    existente = pd.DataFrame(columns=[c for c in novos.columns if c != 'link_norm'])

if 'link_processo' not in existente.columns:
    existente['link_processo'] = ''

# garante coluna manual tamb?m no hist?rico
if 'conclusao_cacs' not in existente.columns:
    existente['conclusao_cacs'] = ''

existente['link_processo'] = existente['link_processo'].astype(str).str.strip()
existente['link_norm'] = existente['link_processo'].apply(normalizar_link)
links_existentes = set(existente['link_norm'].tolist())

novos_filtrados = novos[~novos['link_norm'].isin(links_existentes)].copy()
saida = pd.concat([existente, novos_filtrados], ignore_index=True)

if 'link_norm' in saida.columns:
    saida = saida.drop(columns=['link_norm'])

# reforço final: coluna sempre presente na planilha
if 'conclusao_cacs' not in saida.columns:
    saida['conclusao_cacs'] = ''

with pd.ExcelWriter(ARQUIVO_EXCEL, engine='openpyxl', mode='w') as writer:
    saida.to_excel(writer, sheet_name=ABA_EXCEL, index=False)

print('Arquivo atualizado:', ARQUIVO_EXCEL)
print('Total extraidos nesta execucao:', len(pd.DataFrame(resultados)))
print('Total apos dedupe interno da execucao:', len(novos))
print('Novos adicionados (sem duplicar link):', len(novos_filtrados))
print('Ignorados por duplicidade:', len(novos) - len(novos_filtrados))


# fechamento garantido ao final da execucao
if FECHAR_NAVEGADOR_AUTO:
    encerrar_driver(driver)


Arquivo atualizado: C:\Users\tc832880\Tribunal de Contas do Estado do Paraná\CGF 2025 26 - TramiteScraping\processos_tramite_cacs.xlsx
Total extraidos nesta execucao: 12
Total apos dedupe interno da execucao: 10
Novos adicionados (sem duplicar link): 2
Ignorados por duplicidade: 8
Driver finalizado via quit().
Fallback taskkill executado para PID 31736.


In [ ]:
# --- Encerramento ---

if FECHAR_NAVEGADOR_AUTO:
    # fallback caso a etapa 5 nao tenha sido executada
    try:
        encerrar_driver(driver)
    except Exception as e:
        print('Nao foi possivel encerrar no fallback:', e)
else:
    print('Driver mantido aberto (FECHAR_NAVEGADOR_AUTO=False).')


Driver finalizado via quit().
Fallback taskkill executado para PID 31736.
